In [1]:
import json
import pandas as pd
import numpy as np
import os
import datetime

from src.project_paths import pp
from src.db import DbConnection
from src.server.models.dashboard.surveys.survey_constants import sc
from src.server.models.dashboard.surveys.calculate_dcm_themes_overlap import calculate_dcm_themes_overlap
from src.server.models.dms_model import DmsModel

In [2]:
survey_def = {
    sc.CHARTS: [
        {'id': 'genres', 'name': 'Genres'},
        {'id': 'themes', 'name': 'Themes'},
        {'id': 'behaviors', 'name': 'Behaviors'},
        {'id': 'needs', 'name': 'Needs'},
        {'id': 'competitors', 'name': 'Played Games'},
        {'id': 'best_game', 'name': 'Best Game'},
        {'id': 'routine', 'name': 'Routine'},
        {'id': 'movies', 'name': 'Movies'},
        {'id': 'hobbies', 'name': 'Hobbies'},
        {'id': 'socials', 'name': 'Socials'}],
    sc.SPENDING: [{'id': '1', 'from': '0', 'to': '0', 'group': '1', 'name': '$0'},
 {'id': '2', 'from': '1', 'to': '10', 'group': '2', 'name': '$1 - $10'},
 {'id': '3', 'from': '11', 'to': '20', 'group': '2', 'name': '$11 - $20'},
 {'id': '4', 'from': '21', 'to': '50', 'group': '3', 'name': '$21 - $50'},
 {'id': '5', 'from': '51', 'to': '100', 'group': '3', 'name': '$51 - $100'},
 {'id': '6', 'from': '101', 'to': '500', 'group': '3', 'name': '$101 - $500'},
 {'id': '7', 'from': '501', 'to': '999', 'group': '4', 'name': '$501 - $1000'},
 {'id': '8', 'from': '1000', 'to': '1000', 'group': '4', 'name': '$1000+'}],
    sc.SPENDING_GROUPS: [{'id': '1', 'name': 'Non-payers'},
 {'id': '2', 'name': 'Minnows'},
 {'id': '3', 'name': 'Dolphins'},
 {'id': '4', 'name': 'Whales'}],
    sc.GENRES: [
 {'id': 'puzzle', 'name': 'Puzzles'},
 {'id': 'action', 'name': 'Action'},
 {'id': 'simulation', 'name': 'Simulations'},
 {'id': 'slots', 'name': 'Slots'},
 {'id': 'interactive_story', 'name': 'Interactive story'},
 {'id': 'customization', 'name': 'Customization'},
 {'id': 'strategy', 'name': 'Strategy'},
 {'id': 'rpg', 'name': 'Role Playing Games'},
 {'id': 'ar', 'name': 'Augmented Reality / Location'}],
    sc.GENRES_SUB: [{'id': 'match3', 'name': 'Matching or Merge puzzles', 'genre': 'puzzle'},
 {'id': 'cards', 'name': 'Word or Card puzzles', 'genre': 'puzzle'},
 {'id': 'hidden_objects', 'name': 'Hidden object puzzles', 'genre': 'puzzle'},
 {'id': 'puzzle_physics', 'name': 'Puzzles with physics', 'genre': 'puzzle'},
 {'id': 'labyrinth', 'name': 'Escape or Maze puzzles', 'genre': 'puzzle'},
 {'id': 'non_shooting_action',
  'name': 'Non-shooter Action',
  'genre': 'action'},
 {'id': 'shooting_action', 'name': 'Shooter Action', 'genre': 'action'},
 {'id': 'tycoon', 'name': 'Tycoon', 'genre': 'simulation'},
 {'id': 'idle_tycoon', 'name': 'Idle tycoon', 'genre': 'simulation'},
 {'id': 'life_sim', 'name': 'Life simulations', 'genre': 'simulation'},
 {'id': 'sandbox', 'name': 'Sandbox', 'genre': 'simulation'}],
    sc.THEMES: [{'id': 'adventure', 'name': 'Adventure'}, {'id': 'animals', 'name': 'Animals'}, {'id': 'architecture', 'name': 'Architecture'}, {'id': 'business', 'name': 'Business'}, {'id': 'cars', 'name': 'Cars'}, {'id': 'city', 'name': 'City'}, {'id': 'crime', 'name': 'Crime'}, {'id': 'family', 'name': 'Family'}, {'id': 'fantasy', 'name': 'Fantasy'}, {'id': 'farm', 'name': 'Farm'}, {'id': 'fashion', 'name': 'Fashion'}, {'id': 'food', 'name': 'Food'}, {'id': 'garden', 'name': 'Garden'}, {'id': 'history', 'name': 'History'}, {'id': 'interior', 'name': 'Interior'}, {'id': 'luxury', 'name': 'Luxury'}, {'id': 'military', 'name': 'Military'}, {'id': 'music', 'name': 'Music'}, {'id': 'mystery', 'name': 'Mystery'}, {'id': 'nature', 'name': 'Nature'}, {'id': 'realism', 'name': 'Realism'}, {'id': 'romance', 'name': 'Romance'}, {'id': 'scifi', 'name': 'Sci-fi'}, {'id': 'sport', 'name': 'Sports'}, {'id': 'survival', 'name': 'Survival'}, {'id': 'tech', 'name': 'Technology'}, {'id': 'vehicles', 'name': 'Vehicles'}, {'id': 'weapons', 'name': 'Weapons'}],
    sc.TOPICS: [{'id': 'adventure__discover', 'theme': 'adventure', 'name': 'Discover'}, {'id': 'adventure__travel', 'theme': 'adventure', 'name': 'Travel'}, {'id': 'adventure__survive', 'theme': 'adventure', 'name': 'Survive'}, {'id': 'adventure__connect', 'theme': 'adventure', 'name': 'Connect'}, {'id': 'animals__pets', 'theme': 'animals', 'name': 'Pets'}, {'id': 'animals__exotic', 'theme': 'animals', 'name': 'Exotic'}, {'id': 'animals__marine', 'theme': 'animals', 'name': 'Marine'}, {'id': 'animals__prehistoric', 'theme': 'animals', 'name': 'Prehistoric'}, {'id': 'business__finance', 'theme': 'business', 'name': 'Finance'}, {'id': 'business__corporations', 'theme': 'business', 'name': 'Corporations'}, {'id': 'business__real_estate', 'theme': 'business', 'name': 'Real Estate'}, {'id': 'business__startups', 'theme': 'business', 'name': 'Startups'}, {'id': 'cars__sport', 'theme': 'cars', 'name': 'Sport'}, {'id': 'cars__luxury', 'theme': 'cars', 'name': 'Luxury'}, {'id': 'cars__personal', 'theme': 'cars', 'name': 'Personal'}, {'id': 'cars__delivery', 'theme': 'cars', 'name': 'Delivery'}, {'id': 'crime__police', 'theme': 'crime', 'name': 'Police'}, {'id': 'crime__gangs', 'theme': 'crime', 'name': 'Gangs'}, {'id': 'crime__prison', 'theme': 'crime', 'name': 'Prison'}, {'id': 'crime__drugs', 'theme': 'crime', 'name': 'Drugs'}, {'id': 'family__generations', 'theme': 'family', 'name': 'Generations'}, {'id': 'family__kids', 'theme': 'family', 'name': 'Kids'}, {'id': 'family__heritage', 'theme': 'family', 'name': 'Heritage'}, {'id': 'family__origin', 'theme': 'family', 'name': 'Origin'}, {'id': 'fantasy__sword_magic', 'theme': 'fantasy', 'name': 'Sword & magic'}, {'id': 'fantasy__superheroes', 'theme': 'fantasy', 'name': 'Superheroes'}, {'id': 'fantasy__gods_demons', 'theme': 'fantasy', 'name': 'Gods & demons'}, {'id': 'fantasy__fables', 'theme': 'fantasy', 'name': 'Fables'}, {'id': 'fashion__clothes', 'theme': 'fashion', 'name': 'Clothes'}, {'id': 'fashion__accessories', 'theme': 'fashion', 'name': 'Accessories'}, {'id': 'fashion__cosmetics', 'theme': 'fashion', 'name': 'Cosmetics'}, {'id': 'fashion__shoes', 'theme': 'fashion', 'name': 'Shoes'}, {'id': 'food__recipes', 'theme': 'food', 'name': 'Recipes'}, {'id': 'food__sweets', 'theme': 'food', 'name': 'Sweets'}, {'id': 'food__grill', 'theme': 'food', 'name': 'Grill'}, {'id': 'food__healthy', 'theme': 'food', 'name': 'Healthy'}, {'id': 'history__medieval', 'theme': 'history', 'name': 'Medieval'}, {'id': 'history__ancient', 'theme': 'history', 'name': 'Ancient'}, {'id': 'history__industrial', 'theme': 'history', 'name': 'Industrial'}, {'id': 'history__prehistoric', 'theme': 'history', 'name': 'Prehistoric'}, {'id': 'interior__home', 'theme': 'interior', 'name': 'Home'}, {'id': 'interior__style', 'theme': 'interior', 'name': 'Style'}, {'id': 'interior__historic', 'theme': 'interior', 'name': 'Historic'}, {'id': 'interior__corporate', 'theme': 'interior', 'name': 'Corporate'}, {'id': 'luxury__jewellery', 'theme': 'luxury', 'name': 'Jewellery'}, {'id': 'luxury__real_estate', 'theme': 'luxury', 'name': 'Real estate'}, {'id': 'luxury__yachts', 'theme': 'luxury', 'name': 'Yachts'}, {'id': 'luxury__cars', 'theme': 'luxury', 'name': 'Cars'}, {'id': 'military__battles', 'theme': 'military', 'name': 'Battles'}, {'id': 'military__history', 'theme': 'military', 'name': 'History'}, {'id': 'military__heroes', 'theme': 'military', 'name': 'Heroes'}, {'id': 'military__espionage', 'theme': 'military', 'name': 'Espionage'}, {'id': 'mystery__crime', 'theme': 'mystery', 'name': 'Crime'}, {'id': 'mystery__conspiracy', 'theme': 'mystery', 'name': 'Conspiracy'}, {'id': 'mystery__supernatural', 'theme': 'mystery', 'name': 'Supernatural'}, {'id': 'mystery__scientific', 'theme': 'mystery', 'name': 'Scientific'}, {'id': 'romance__love', 'theme': 'romance', 'name': 'Love'}, {'id': 'romance__relationships', 'theme': 'romance', 'name': 'Relationships'}, {'id': 'romance__sex', 'theme': 'romance', 'name': 'Sex'}, {'id': 'romance__youth', 'theme': 'romance', 'name': 'Youth'}, {'id': 'sport__team', 'theme': 'sport', 'name': 'Team'}, {'id': 'sport__individual', 'theme': 'sport', 'name': 'Individual'}, {'id': 'sport__fitness', 'theme': 'sport', 'name': 'Fitness'}, {'id': 'sport__martial_arts', 'theme': 'sport', 'name': 'Martial arts'}, {'id': 'survival__zombies', 'theme': 'survival', 'name': 'Zombies'}, {'id': 'survival__wilderness', 'theme': 'survival', 'name': 'Wilderness'}, {'id': 'survival__disaster', 'theme': 'survival', 'name': 'Disaster'}, {'id': 'survival__war', 'theme': 'survival', 'name': 'War'}, {'id': 'tech__electronics', 'theme': 'tech', 'name': 'Electronics'}, {'id': 'tech__software', 'theme': 'tech', 'name': 'Software'}, {'id': 'tech__mechanics', 'theme': 'tech', 'name': 'Mechanics'}, {'id': 'tech__medicine', 'theme': 'tech', 'name': 'Medicine'}, {'id': 'vehicles__motorcycles', 'theme': 'vehicles', 'name': 'Motorcycles'}, {'id': 'vehicles__airplanes', 'theme': 'vehicles', 'name': 'Airplanes'}, {'id': 'vehicles__trains', 'theme': 'vehicles', 'name': 'Trains'}, {'id': 'vehicles__ships', 'theme': 'vehicles', 'name': 'Ships'}, {'id': 'weapons__guns', 'theme': 'weapons', 'name': 'Guns'}, {'id': 'weapons__high_tech', 'theme': 'weapons', 'name': 'High-tech'}, {'id': 'weapons__mass_destruction', 'theme': 'weapons', 'name': 'Mass destruction'}, {'id': 'weapons__historic', 'theme': 'weapons', 'name': 'Historic'}, {'id': 'nature__islands', 'theme': 'nature', 'name': 'Islands'}, {'id': 'nature__animals', 'theme': 'nature', 'name': 'Animals'}, {'id': 'nature__mountains', 'theme': 'nature', 'name': 'Mountains'}, {'id': 'nature__water', 'theme': 'nature', 'name': 'Water'}, {'id': 'realism__life', 'theme': 'realism', 'name': 'Life'}, {'id': 'realism__people', 'theme': 'realism', 'name': 'People'}, {'id': 'realism__truth', 'theme': 'realism', 'name': 'Truth'}, {'id': 'realism__document', 'theme': 'realism', 'name': 'Document'}, {'id': 'scifi__universe', 'theme': 'scifi', 'name': 'Universe'}, {'id': 'scifi__aliens', 'theme': 'scifi', 'name': 'Aliens'}, {'id': 'scifi__future', 'theme': 'scifi', 'name': 'Future'}, {'id': 'scifi__technology', 'theme': 'scifi', 'name': 'Technology'}, {'id': 'architecture__building', 'theme': 'architecture', 'name': 'Building'}, {'id': 'architecture__design', 'theme': 'architecture', 'name': 'Design'}, {'id': 'architecture__construction', 'theme': 'architecture', 'name': 'Construction'}, {'id': 'architecture__art', 'theme': 'architecture', 'name': 'Art'}, {'id': 'city__building', 'theme': 'city', 'name': 'building'}, {'id': 'city__people', 'theme': 'city', 'name': 'people'}, {'id': 'city__busy', 'theme': 'city', 'name': 'busy'}, {'id': 'city__traffic', 'theme': 'city', 'name': 'traffic'}, {'id': 'farm__animals', 'theme': 'farm', 'name': 'animals'}, {'id': 'farm__crops', 'theme': 'farm', 'name': 'crops'}, {'id': 'farm__trees', 'theme': 'farm', 'name': 'trees'}, {'id': 'farm__garden', 'theme': 'farm', 'name': 'garden'}, {'id': 'garden__flowers', 'theme': 'garden', 'name': 'flowers'}, {'id': 'garden__plants', 'theme': 'garden', 'name': 'plants'}, {'id': 'garden__trees', 'theme': 'garden', 'name': 'trees'}, {'id': 'garden__vegetables', 'theme': 'garden', 'name': 'vegetables'}, {'id': 'music__dance', 'theme': 'music', 'name': 'dance'}, {'id': 'music__sing', 'theme': 'music', 'name': 'sing'}, {'id': 'music__chill', 'theme': 'music', 'name': 'chill'}, {'id': 'music__background', 'theme': 'music', 'name': 'background'}],
    sc.BEHAVIORS: [{'id': 'grow_build', 'name': 'Grow (build)', 'desc': 'Building and managing cities, parks or companies'}, {'id': 'grow_power', 'name': 'Grow (power)', 'desc': 'Growing / powering up your game objects, characters or items'}, {'id': 'think_plan', 'name': 'Solve (plan)', 'desc': 'Planning or deciding the best way to achieve something'}, {'id': 'think_react', 'name': 'Solve (react)', 'desc': 'Solving challenges or puzzles'}, {'id': 'action_high', 'name': 'Action (high)', 'desc': 'Fast-paced and intense gameplay'}, {'id': 'dominate', 'name': 'Dominate', 'desc': 'Going up against others in battles or matches'}, {'id': 'immerse_characters', 'name': 'Immerse (chars)', 'desc': 'Interesting characters in a game world or story'}, {'id': 'immerse_story', 'name': 'Immerse (story)', 'desc': 'Following a game story'}, {'id': 'thrill', 'name': 'Thrill', 'desc': 'Destroying game environment or enemies'}, {'id': 'discover', 'name': 'Discover', 'desc': 'Exploring the game world or content'}, {'id': 'connect_coop', 'name': 'Connect (co-op)', 'desc': 'Cooperating with others'}, {'id': 'connect_compete', 'name': 'Connect (compete)', 'desc': 'Competing with others'}, {'id': 'difficulty_high', 'name': 'Difficulty (high)', 'desc': 'Practicing and mastering a difficult gameplay'}, {'id': 'care', 'name': 'Care', 'desc': 'Helping or caring for others'}, {'id': 'collect', 'name': 'Collect', 'desc': 'Collecting in-game items or valuables'}, {'id': 'experiment', 'name': 'Experiment', 'desc': 'Open game where you can experiment or do what you want'}, {'id': 'express', 'name': 'Express', 'desc': 'Designing and decorating interior / exterior or a character'}, {'id': 'complete', 'name': 'Complete', 'desc': 'Completing various tasks or levels'}, {'id': 'action_low', 'name': 'Action (low)', 'desc': 'Contemplative and calm gameplay'}, {'id': 'difficulty_low', 'name': 'Difficulty (low)', 'desc': 'Relaxing and enjoying a simple gameplay'}],
    sc.NEEDS: [{'id': 'insight', 'name': 'Insight', 'desc': 'To get knowledge'}, {'id': 'mastery', 'name': 'Mastery', 'desc': 'To train my skills (any)'}, {'id': 'novelty', 'name': 'Novelty', 'desc': 'To have a variety of content'}, {'id': 'uniqueness', 'name': 'Uniqueness', 'desc': 'To get something rare'}, {'id': 'accomplishment', 'name': 'Accomplishment', 'desc': 'To accomplish something great'}, {'id': 'ownership', 'name': 'Ownership', 'desc': 'To own something by myself'}, {'id': 'wealth', 'name': 'Wealth', 'desc': 'To grow value of something'}, {'id': 'control', 'name': 'Control', 'desc': 'To be in control'}, {'id': 'safety', 'name': 'Safety', 'desc': 'To protect something'}, {'id': 'fitness', 'name': 'Fitness', 'desc': 'To perform any physical activity'}, {'id': 'relief', 'name': 'Relief', 'desc': 'To relax'}, {'id': 'recognition', 'name': 'Recognition', 'desc': 'To be known'}, {'id': 'acceptance', 'name': 'Acceptance', 'desc': 'To be part of the group'}, {'id': 'morality', 'name': 'Morality', 'desc': 'To help / care for others'}, {'id': 'superiority', 'name': 'Superiority', 'desc': 'To be exceptional'}, {'id': 'respect', 'name': 'Respect', 'desc': 'To be followed by others'}, {'id': 'trust', 'name': 'Trust', 'desc': 'To be able to rely on someone'}, {'id': 'intimacy', 'name': 'Intimacy', 'desc': 'To experience romantic / intimate relationship'}],
    sc.PLAYING: [{'id': '1', 'name': 'Less than 1 hour'},
 {'id': '2', 'name': '1 – 5 hours'},
 {'id': '3', 'name': '6 – 10 hours'},
 {'id': '4', 'name': '11 – 15 hours'},
 {'id': '5', 'name': 'More than 15 hours'},
 {'id': '6', 'name': "Don't know"}],
    sc.ROUTINE: [{'id': 'working', 'name': 'Working'}, {'id': 'hobbies', 'name': 'Engaging in hobbies'}, {'id': 'reading', 'name': 'Reading'}, {'id': 'social_media', 'name': 'Using social media'}, {'id': 'following_news', 'name': 'Following news'}, {'id': 'learning', 'name': 'Learning new things'}, {'id': 'music', 'name': 'Listening to music'}, {'id': 'movies', 'name': 'Watching movies / shows'}, {'id': 'partner', 'name': 'Spending time with partner'}, {'id': 'friends', 'name': 'Spending time with friends'}, {'id': 'parenting', 'name': 'Parenting'}, {'id': 'pets', 'name': 'Taking care of pets'}, {'id': 'video_games', 'name': 'Playing video games'}],
    sc.HOBBIES: [{'id': 'collecting', 'name': 'Collecting'}, {'id': 'design', 'name': 'Designing / engineering'}, {'id': 'teaching', 'name': 'Teaching / training others'}, {'id': 'serving_community', 'name': 'Serving community / society'}, {'id': 'fishing_hunting', 'name': 'Fishing / hunting'}, {'id': 'farm_garden', 'name': 'Farming / gardening'}, {'id': 'performing_arts', 'name': 'Performing arts'}, {'id': 'driving', 'name': 'Driving / maintaining vehicles'}, {'id': 'building_fixing', 'name': 'Building / fixing things'}, {'id': 'cooking', 'name': 'Cooking / preparing meals'}, {'id': 'exercising_sports', 'name': 'Exercising / performing sports'}, {'id': 'photographing', 'name': 'Photography'}, {'id': 'DIY', 'name': 'DIY / Decorating interiors'}, {'id': 'gardening', 'name': 'Landscaping / gardening'}, {'id': 'watching_games', 'name': 'Watching others play games'}, {'id': 'watching_sports', 'name': 'Watching classic sports'}],
    sc.MOVIES: [{'id': 'action_adventure', 'name': 'Action / adventure'}, {'id': 'anime', 'name': 'Anime'}, {'id': 'children_family', 'name': 'Children / family'}, {'id': 'classic', 'name': 'Classic'}, {'id': 'comedies', 'name': 'Comedies'}, {'id': 'criminal', 'name': 'Criminal'}, {'id': 'documentary', 'name': 'Documentary'}, {'id': 'drama', 'name': 'Drama'}, {'id': 'horror', 'name': 'Horror'}, {'id': 'romantic', 'name': 'Romantic'}, {'id': 'fantasy', 'name': 'Fantasy'}, {'id': 'scifi', 'name': 'Sci-fi'}, {'id': 'thriller', 'name': 'Thriller'}],
    sc.SOCIALS: [{'id': 'facebook', 'name': 'Facebook'}, {'id': 'Instagram', 'name': 'instagram'}, {'id': 'youtube', 'name': 'Youtube'}, {'id': 'whatsapp', 'name': 'Whatsapp'}, {'id': 'pinterest', 'name': 'Pinterest'}, {'id': 'twitter', 'name': 'Twitter'}, {'id': 'snapchat', 'name': 'Snapchat'}, {'id': 'discord', 'name': 'Discord'}, {'id': 'twitch', 'name': 'Twitch'}, {'id': 'reddit', 'name': 'Reddit'}, {'id': 'tiktok', 'name': 'TikTok'}, {'id': 'telegram', 'name': 'Telegram'}],
    sc.COMPETITORS: [{'id': 'lumber_inc', 'name': 'Lumber INC'}, {'id': 'idle_miner_tycoon', 'name': 'Idle Miner Tycoon'}, {'id': 'prison_empire_tycoon', 'name': 'Prison Empire Tycoon'}, {'id': 'adventure_capitalist', 'name': 'Adventure Capitalist'}, {'id': 'idle_theme_park_tycoon', 'name': 'Idle Theme Park Tycoon'}, {'id': 'klondike_adventures', 'name': 'Klondike Adventures'}, {'id': 'family_island', 'name': 'Family Island'}, {'id': 'airport_city', 'name': 'Airport City'}, {'id': 'township', 'name': 'Township'}, {'id': 'simcity_buildit', 'name': 'SimCity BuildIt'}, {'id': 'megapolis', 'name': 'Megapolis'}, {'id': 'emergency_hq', 'name': 'Emergency HQ'}, {'id': 'hay_day', 'name': 'Hay Day'}, {'id': 'dragon_city', 'name': 'Dragon City'}, {'id': 'junes_journey', 'name': "June's Journey"}, {'id': 'pearls_peril', 'name': "Pearl's Peril"}, {'id': 'seekers_notes', 'name': "Seeker's Notes"}, {'id': 'murder_in_alps_hidden_mystery', 'name': 'Murder in Alps: Hidden Mystery'}, {'id': 'lilys_garden', 'name': "Lily's Garden"}, {'id': 'candy_crush_saga', 'name': 'Candy Crush Saga'}, {'id': 'gardenscapes', 'name': 'Gardenscapes'}, {'id': 'match_masters', 'name': 'Match Masters'}, {'id': 'royal_match', 'name': 'Royal Match'}, {'id': 'toon_blast', 'name': 'Toon Blast'}, {'id': 'project_makeover', 'name': 'Project Makeover'}, {'id': 'merge_mansion', 'name': 'Merge Mansion'}, {'id': 'merge_dragons', 'name': 'Merge Dragons'}, {'id': 'ever_merge', 'name': 'Ever Merge'}, {'id': 'wordscapes', 'name': 'Wordscapes'}, {'id': 'solitaire_grand_harvest', 'name': 'Solitaire Grand Harvest'}, {'id': 'angry_birds_2', 'name': 'Angry Birds 2'}, {'id': 'cut_the_rope', 'name': 'Cut the Rope'}, {'id': 'world_of_goo', 'name': 'World of Goo'}, {'id': 'monument_valley', 'name': 'Monument Valley'}, {'id': 'lara_croft_go', 'name': 'Lara Croft GO'}, {'id': 'diggys_adventure', 'name': "Diggy's Adventure"}, {'id': 'the_room', 'name': 'The Room'}, {'id': 'evony', 'name': 'Evony'}, {'id': 'clash_of_clans', 'name': 'Clash of Clans'}, {'id': 'king_of_avalon', 'name': 'King of Avalon'}, {'id': 'state_of_survival', 'name': 'State of Survival'}, {'id': 'rise_of_kingdoms', 'name': 'Rise of Kingdoms'}, {'id': 'clash_royale', 'name': 'Clash Royale'}, {'id': 'brawl_stars', 'name': 'Brawl Stars'}, {'id': 'archero', 'name': 'Archero'}, {'id': 'pubg_mobile', 'name': 'PUBG Mobile'}, {'id': 'cats_crash_arena_turbo_stars', 'name': 'CATS: Crash Arena Turbo Stars'}, {'id': 'pokemon_go', 'name': 'Pokemon GO'}, {'id': 'episode_choose_your_story', 'name': 'Episode - Choose Your Story'}, {'id': 'chapters_interactive_stories', 'name': 'Chapters: Interactive Stories'}, {'id': 'the_sims', 'name': 'The Sims'}, {'id': 'avakin_life_3d', 'name': 'Avakin Life 3D'}, {'id': 'idle_life_sim', 'name': 'Idle Life Sim'}],
    sc.DCM_TITLE: [{'id': 'none', 'name': 'none', 'desc': 'none'}, {'id': '1', 'name': 'Family Adventures', 'desc': 'Complete quests at a house with friendly local characters.'}, {'id': '2', 'name': 'Time-travel Adventures', 'desc': 'Complete quests at various worlds with different civilizations.'}, {'id': '3', 'name': 'Survival Adventures', 'desc': 'Complete quests in a world with zombies.'}, {'id': '4', 'name': 'Celestial Adventures', 'desc': 'Complete quests in various worlds with different religious figures.'}, {'id': '5', 'name': 'Holiday Adventures', 'desc': 'Complete quests at various exotic locations with different locals.'}, {'id': '6', 'name': 'Detective Adventures', 'desc': 'Complete cases in a world with criminals and victims.'}, {'id': '7', 'name': 'Mafia Adventures', 'desc': 'Complete quests in a world with gangs and businesspeople.'}, {'id': '8', 'name': 'Mystery Adventures', 'desc': 'Complete cases in a world with supernatural creatures.'}, {'id': '9', 'name': 'Sci-Fi Adventures', 'desc': 'Complete quests at various planets with different civilizations.'}, {'id': '10', 'name': 'Historic Adventures', 'desc': 'Complete quests in various historical events with different characters.'}, {'id': '11', 'name': 'Army Adventures', 'desc': 'Complete quests in a world with soldiers.'}, {'id': '12', 'name': 'Harry Potter Adventures', 'desc': 'Complete quests in a world with wizards and beasts.'}, {'id': '13', 'name': 'Atlantis Adventures', 'desc': 'Complete quests at the island with mythical civilization.'}, {'id': '14', 'name': 'Superhero Adventures', 'desc': 'Complete quests in a world with superheroes and supervillains.'}, {'id': '15', 'name': 'Celebrity Adventures', 'desc': 'Complete quests in a city with popular characters.'}, {'id': '16', 'name': 'Legendary Adventures', 'desc': 'Complete quests in various worlds with mythical creatures.'}, {'id': '17', 'name': 'Country Adventures', 'desc': 'Complete quests in a town with local characters.'}],
    sc.DCM_FEATURES: [{'id': '1', 'name': 'Solve puzzles with physics', 'group': 1}, {'id': '2', 'name': 'Solve hidden object puzzles', 'group': 1}, {'id': '3', 'name': 'Solve word or card puzzles', 'group': 1}, {'id': '4', 'name': 'Solve escape or maze puzzles', 'group': 1}, {'id': '5', 'name': 'Solve match or merge puzzles', 'group': 1}, {'id': '6', 'name': 'Help game characters', 'group': 2}, {'id': '7', 'name': 'Destroy game enemies', 'group': 2}, {'id': '8', 'name': 'Puzzles are challenging', 'group': 2}, {'id': '9', 'name': 'Puzzles are simple', 'group': 2}, {'id': '10', 'name': 'Explore various worlds', 'group': 2}, {'id': '11', 'name': 'Equip your game character', 'group': 3}, {'id': '12', 'name': 'Play with other players', 'group': 3}, {'id': '13', 'name': 'Follow dramatic stories', 'group': 3}, {'id': '14', 'name': 'Build or renovate your place', 'group': 3}, {'id': '15', 'name': 'Follow funny stories', 'group': 3}]
}

In [3]:
def load_panel_csv(csv_path_1, csv_path_2):
    df = pd.read_csv(csv_path_1, dtype={
        'Status': str,
        'URL Variable: transaction_id': 'Int64',
        'URL Variable: sguid': str,
        'Other - Write In:In the previous answer you mentioned using social media. Which social platforms do you use the most?.5': str,
        'Other - Write In:In the previous answer you mentioned using social media. Which social platforms do you use the most?.2': str})

    df = df[df['Status'] == 'Complete']

    df['spending'] = df['In the past 3 months, how much would you say you have spent within mobile games?']

    df_panel = pd.read_csv(csv_path_2, dtype={'TRANSID': 'Int64', 'Code': str, 'Year': 'Int64', "'Age'": 'Int64'})

    df = pd.merge(df, df_panel, how='inner', left_on=['URL Variable: transaction_id'], right_on=['TRANSID'])

    return df

In [4]:
def load_dcm_csv(csv_path):
    df = pd.read_csv(csv_path, dtype={
        'Response ID': 'Int64',
        'URL Variable: transaction_id': 'Int64',
        'URL Variable: sguid': str,
        'Other - Write In:In the previous answer you mentioned using social media. Which social platforms do you use the most?.5': str,
        'Other - Write In:In the previous answer you mentioned using social media. Which social platforms do you use the most?.2': str})

    return df

In [5]:
def create_panel_df(df: pd.DataFrame, needs_0_1=False):
    df_final = pd.DataFrame()
    df_final['guid'] = df['Response ID']
    df_final['cint_id'] = df_final['guid']
    df_final['start'] = df['Time Started']
    df_final['end'] = df['Date Submitted']
    df_final['time'] = 0
    def convert_time(r):
        l_end = datetime.datetime.strptime(r['end'], "%d/%m/%Y %I:%M:%S %p").timestamp()
        l_start = datetime.datetime.strptime(r['start'], "%d/%m/%Y %I:%M:%S %p").timestamp()
        r['time'] = l_end - l_start
        return r
    df_final = df_final.apply(lambda r: convert_time(r), axis=1)
    df_final.drop(['start', 'end'], axis=1, inplace=True)

    # age
    df_final['age'] = df["'Age'"]

    # gender
    df_final.loc[df["'Gender'"] == 'Male', 'gender'] = 'm'
    df_final.loc[df["'Gender'"] == 'Female', 'gender'] = 'f'

    # spending
    def convert_spending(r):
        for it in survey_def[sc.SPENDING]:
            n: str = it['name']
            n = n.replace(" ", "")
            if n == r['spending']:
                r['spending'] = it['id']
                if it['from'] == 0:
                    r['spending__avg'] = 0
                else:
                    r['spending__avg'] = round((int(it['to']) + int(it['from'])) / 2, 2)
                return r
        return r

    df_final['spending'] = df['spending']
    df_final = df_final.apply(lambda r: convert_spending(r), axis=1)

    # genres, subgenres
    genre_question = 'How do you feel about the following types'
    genre_answers = ["I don’t know this type",
         "I don’t like these games",
         'These games are okay',
         'I enjoy these games',
         'I love these games']

    for col in df.columns:
        if genre_question in col:
            for g_def in survey_def[sc.GENRES]:
                if g_def['name'] in col:
                    df_final[f"{sc.get_prefix(sc.GENRES)}{g_def['id']}"] = df[col]
    for col in df.columns:
        if genre_question in col:
            for g_def in survey_def[sc.GENRES_SUB]:
                if g_def['name'] in col:
                    df_final[f"{sc.get_prefix(sc.GENRES_SUB)}{g_def['id']}"] = df[col]
    for col in df_final.columns:
        if sc.get_prefix(sc.GENRES) in col or sc.get_prefix(sc.GENRES_SUB) in col:
            for i in range(len(genre_answers)):
                df_final[col].replace(genre_answers[i], 0.25 * i, inplace=True)

    # themes, topics
    themes_question = 'Which topics / themes would you like in an ideal mobile game?'
    topics_question = 'Which of the topics related to the theme'
    topics_open_question = 'write down three words you associate with the theme'

    for col in df.columns:
        if themes_question in col:
            for g_def in survey_def[sc.THEMES]:
                if g_def['name'] in col:
                    fcol = f"{sc.get_prefix(sc.THEMES)}{g_def['id']}"
                    df_final[fcol] = df[col]
                    df_final[fcol].replace(g_def['name'], 1, inplace=True)
                    df_final[fcol].replace(np.NaN, 0, inplace=True)

    df_final = df_final.copy()

    for col in df.columns:
        if topics_question in col:
            for d in survey_def[sc.THEMES]:
                if d['name'] in col:
                    for td in survey_def[sc.TOPICS]:
                        if td['theme'] == d['id']:
                            fcol = f"{sc.get_prefix(sc.TOPICS)}{td['id']}"
                            df_final[fcol] = 0
                            df_final.loc[df[col] == td['name'], fcol] = 1

    df_final = df_final.copy()

    # needs
    needs_question = 'What would you want from an ideal mobile game?'
    needs_answers = ['Not important at all','Less important','Somehow important','Very important']
    for col in df.columns:
        for needs_question in col:
            for it in survey_def[sc.NEEDS]:
                if it['desc'] in col:
                    fcol = f"{sc.get_prefix(sc.NEEDS)}{it['id']}"
                    df_final[fcol] = df[col]
                    if needs_0_1:
                        df_final[fcol].replace(np.NaN, 0, inplace=True)
                        df_final[fcol].replace(it['desc'], 1, inplace=True)
                    else:
                        for i in range(len(needs_answers)):
                            df_final[fcol].replace(needs_answers[i], 1 if i == 3 else 0.33 * i, inplace=True)

    # behaviours
    behaviours_question = 'enjoy the following elements'
    behaviours_answers_v1 = ["I don’t like this","It’s okay as a side element","I enjoy this"]
    behaviours_answers_v2 = ["I don’t like this","It's ok as a side element","I enjoy this"]
    for col in df.columns:
        for behaviours_question in col:
            for it in survey_def[sc.BEHAVIORS]:
                if it['desc'] in col:
                    fcol = f"{sc.get_prefix(sc.BEHAVIORS)}{it['id']}"
                    df_final[fcol] = df[col]
                    for i in range(len(behaviours_answers_v1)):
                        df_final[fcol].replace(behaviours_answers_v1[i], 0.5 * i, inplace=True)
                        df_final[fcol].replace(behaviours_answers_v2[i], 0.5 * i, inplace=True)


    # playing
    playing_question = 'In a typical week, how much time do you spend playing mobile games?'
    for col in df.columns:
        if playing_question in col:
            df_final['playing'] = df[col]
            df_final['playing__avg'] = 0
            for it in survey_def[sc.PLAYING]:
                df_final['playing'].replace(it['name'], it['id'], inplace=True)
            break

    # best game
    best_game_question = 'What is your most favorite mobile game of all time?'
    for col in df.columns:
        if best_game_question in col:
            df_final['best_game'] = df[col]
            break

    # routine, movies, hobbies, socials
    routine_question = 'Which of the following activities are part of your regular routine?'
    for col in df.columns:
        if routine_question in col:
            for it in survey_def[sc.ROUTINE]:
                if it['name'] in col:
                    fcol = f"{sc.get_prefix(sc.ROUTINE)}{it['id']}"
                    df_final[fcol] = df[col]
                    df_final[fcol].replace(it['name'], 1, inplace=True)
                    df_final[fcol].replace(np.NaN, 0, inplace=True)

    hobbies_question = 'In the previous answer you mentioned you engage in hobbies'
    for col in df.columns:
        if hobbies_question in col:
            for it in survey_def[sc.HOBBIES]:
                if it['name'] in col:
                    fcol = f"{sc.get_prefix(sc.HOBBIES)}{it['id']}"
                    df_final[fcol] = df[col]
                    df_final[fcol].replace(it['name'], 1, inplace=True)
                    df_final[fcol].replace(np.NaN, 0, inplace=True)

    movies_question = 'In the previous answer you mentioned watching movies / shows. Which movie genres do you like the most?'
    for col in df.columns:
        if movies_question in col:
            for it in survey_def[sc.MOVIES]:
                if it['name'] in col:
                    fcol = f"{sc.get_prefix(sc.MOVIES)}{it['id']}"
                    df_final[fcol] = df[col]
                    df_final[fcol].replace(it['name'], 1, inplace=True)
                    df_final[fcol].replace(np.NaN, 0, inplace=True)

    social_question = 'In the previous answer you mentioned using social media. Which social platforms do you use the most?'
    for col in df.columns:
        if social_question in col:
            for it in survey_def[sc.SOCIALS]:
                if it['name'] in col:
                    fcol = f"{sc.get_prefix(sc.SOCIALS)}{it['id']}"
                    df_final[fcol] = df[col]
                    df_final[fcol].replace(it['name'], 1, inplace=True)
                    df_final[fcol].replace(np.NaN, 0, inplace=True)

    df_final = df_final.copy()

    # competitors
    competitor_question = ':How do you relate to each of these'
    competitor_answers = ['Never heard of it','Never played','Played it a little','Used to play it a lot','Currently play it']
    for col in df.columns:
        if competitor_question in col:
            for it in survey_def[sc.COMPETITORS]:
                if it['name'] in col:
                    fcol = f"{sc.get_prefix(sc.COMPETITORS)}{it['id']}"
                    df_final[fcol] = df[col]
                    for i in range(len(competitor_answers)):
                        df_final[fcol].replace(competitor_answers[i], 0.25 * i, inplace=True)

    df_final = df_final.copy()
    return df_final

In [6]:
def create_dcm_df(df: pd.DataFrame):
    df_final = pd.DataFrame()

    df_final['guid'] = df['Response ID']

    for i in range(len(df.columns)):
        if 'Set Number' in df.columns[i]:
            df_final['set_num'] = df[df.columns[i]]
        elif 'Card Number' in df.columns[i]:
            df_final['card_num'] = df[df.columns[i]]
        elif 'Score' in df.columns[i]:
            df_final['score'] = df[df.columns[i]]

    b_flag_start = '<b>'
    b_flag_end = '</b>'
    br_flag = '<br />'

    def parseParam(value):
        if '</b>' in value:
            return value[len(b_flag_start):value.index(b_flag_end):]
        return value

    def parseDesc(value):
        if br_flag in value:
            index = value.index(br_flag)
            return value[index + len(br_flag):value.index(br_flag, index + 1):]
        return value

    def readParam(param_index):
        tmp = 1
        for i in range(len(df.columns)):
            if 'Please choose' in df.columns[i]:

                if (('Set Number' not in df.columns[i]) & ('Card Number' not in df.columns[i]) & ('Score' not in df.columns[i])):
                    if tmp == param_index:

                        if param_index == 1:
                            df_final[f'desc'] = df[df.columns[i]].apply(lambda value: parseDesc(value))

                        df_final[f'param__{param_index}'] = df[df.columns[i]].apply(lambda value: parseParam(value))

                        return
                    else:
                        tmp = tmp + 1

    readParam(1)
    readParam(2)
    readParam(3)
    readParam(4)

    df_final['t'] = df_final['param__1']
    df_final['f1'] = df_final['param__2']
    df_final['f2'] = df_final['param__3']
    df_final['f3'] = df_final['param__4']

    for it in survey_def[sc.DCM_TITLE]:
        df_final['t'].replace(it['name'], it['id'], inplace=True)
    for it in survey_def[sc.DCM_FEATURES]:
        df_final['f1'].replace(it['name'], it['id'], inplace=True)
        df_final['f1'].replace('none', '0', inplace=True)
        df_final['f2'].replace(it['name'], it['id'], inplace=True)
        df_final['f2'].replace('none', '0', inplace=True)
        df_final['f3'].replace(it['name'], it['id'], inplace=True)
        df_final['f3'].replace('none', '0', inplace=True)

    df_final.drop(['desc', 'param__1', 'param__2', 'param__3', 'param__4'], inplace=True, axis=1)

    df_final = df_final[df_final['card_num'].isin([1, 2, 3, 4])]
    df_final = df_final[df_final['set_num'].isin([1, 2, 3, 4, 5, 6, 7, 8])]
    df_final = df_final[df_final['score'].isin([0, 100])]
    arr = df_final['guid'].value_counts()
    arr = arr[arr.values == 32]
    df_final = df_final[df_final['guid'].isin(arr.index)]

    df_final = df_final.copy()
    df_final.reset_index(inplace=True, drop=True)

    return df_final

In [7]:
def create_survey_db(conn, survey_guid, panel_df, dcm_df, definitions, is_panel, is_dcm):

    survey_config = {}
    survey_config['data_version'] = 1
    survey_config['groups'] = {
        '1': {
            'lang': 'en',
            'quota': '2000',
            "name": "DEMO"
        },
    }
    if dcm_df is not None:
        survey_config['dcm_config'] = {
            'features': 3,
            'choices': 4,
            'set': 8,
            "name": "DEMO"
        }

    conn.query("""
    DELETE FROM app.surveys WHERE guid = %s
    """, [survey_guid])

    conn.query("""
    INSERT INTO app.surveys (dms_id, name, is_active, survey_config, guid, state, is_panel, is_dcm) VALUES
    (%s, %s, %s, %s, %s, %s, %s, %s)
    """, [0, 'DEMO', 0, json.dumps(survey_config), survey_guid, 'survey_data_processed', is_panel, is_dcm])

    if panel_df is not None:
        DmsModel.save_df_to_dms(conn, panel_df, guid=f"{survey_guid}__1__panel")
    if dcm_df is not None:
        DmsModel.save_df_to_dms(conn, dcm_df, guid=f"{survey_guid}__1__dcm")
    DmsModel.save_json_to_dms(conn, definitions, guid=f"{survey_guid}__def")

In [10]:
from src.runtime_constants import rr

conn = DbConnection(rr.ENV_PROD)

In [12]:
conn.close()

In [11]:
# panel
survey_guid = 'd4c8ed7a-158d-4f5a-8294-71f62730f4c1'

csv_path_1 = os.path.join(pp.DATA, "models", "alchemy", 'csv', 'Alch-20220820083331-SurveyExport.csv')
csv_path_2 = os.path.join(pp.DATA, "models", "alchemy", 'csv', 'Prodege_Panel_Export_Motivators2.0-MarketMapv3_20220815-20220818_C.csv')

panel_df = create_panel_df(load_panel_csv(csv_path_1, csv_path_2))

create_survey_db(conn, survey_guid, panel_df, None, survey_def, is_panel=1, is_dcm=0)
conn.commit()

In [9]:
# dcm
survey_guid = '313a0507-d2fe-4b78-b56b-9f587b97f114'

dcm_csv_path_1 = os.path.join(pp.DATA, "models", "alchemy", 'csv', 'PuzzleDCM_Alch-20220707090057-SurveyExport.csv')
dcm_csv_path_2 = os.path.join(pp.DATA, "models", "alchemy", 'csv', 'PuzzleDCM_Prodege-Export_PuzzlesDCMv2_20220703-20220707_C.csv')
dcm_csv_path_3 = os.path.join(pp.DATA, "models", "alchemy", 'csv', 'PuzzleDCM_Alch-Conjoint-20220707090234-SurveyExport.csv')

dcm_panel_df = create_panel_df(load_panel_csv(dcm_csv_path_1, dcm_csv_path_2), True)
dcm_df = create_dcm_df(load_dcm_csv(dcm_csv_path_3))

create_survey_db(conn, survey_guid, dcm_panel_df, dcm_df, survey_def, is_panel=0, is_dcm=1)

print("CALCULATE THEME OVERLAP - START")

row = conn.select_one("""SELECT survey_config FROM app.surveys WHERE guid = %s""", [survey_guid])
dcm_config = json.loads(row['survey_config'])['dcm_config']
themes_overlap = calculate_dcm_themes_overlap(dcm_panel_df, dcm_df, dcm_config)
DmsModel.save_json_to_dms(conn, themes_overlap, guid=f"{survey_guid}__1__themes_overlap")
conn.commit()
print("CALCULATE THEME OVERLAP - END")

CALCULATE THEME OVERLAP - START


converting data_co to <class 'numpy.float64'>
C:\Work\Projects\Gembase\dashboard_main\dashboard_server\conda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 0.0 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")


CALCULATE THEME OVERLAP - END
